# Main Agent playground

실제 HCX 모델, Domain Agent, Search Service와 Qdrant를 조립해 Main Supervisor의 전체 실행 경로를 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

Main Agent는 질문 하나에 HCX를 여러 번 호출할 수 있다. Runtime은 한 번만 조립해 재사용하고 질문은 순차적으로 실행한다. 노트북을 커밋하기 전에는 검색 원문과 답변을 포함한 모든 출력을 제거한다.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

from pension_agent.agent.runtime import build_runtime_answer_service
from pension_agent.api.presentation import build_answer_response

## 실행 환경 확인

Python 경로가 저장소의 `.venv`를 가리키고 현재 디렉터리에 `.env`가 있어야 한다. API key 값 자체는 출력하지 않는다.

In [ ]:
project_root = Path.cwd()
{
    "python": sys.executable,
    "cwd": str(project_root),
    "env_file_exists": (project_root / ".env").is_file(),
}

## 전체 Agent Runtime 조립

이 셀을 다시 실행하면 기존 Runtime의 외부 client를 먼저 닫고 새 Runtime을 만든다.

In [ ]:
existing_answer_service = globals().get("answer_service")
if existing_answer_service is not None:
    await existing_answer_service.aclose()

answer_service = await build_runtime_answer_service()
"Agent Runtime 준비 완료"

## 질문 실행 helper

평가 API와 동일한 5개 응답 필드와 검증된 내부 DomainResult를 함께 반환한다.

In [ ]:
async def ask_main_agent(*, question_id: str, question: str) -> dict[str, object]:
    result = await answer_service.run(
        question_id=question_id,
        question=question,
    )
    response = build_answer_response(result)
    return {
        "response": response.model_dump(mode="json"),
        "domain_results": result.state["domain_results"],
    }

## 질문 하나 실행

먼저 최종 API 응답을 확인하고, 필요할 때 내부 도메인 결과를 별도로 살펴본다.

In [ ]:
run = await ask_main_agent(
    question_id="NB-001",
    question="IRP와 연금저축의 차이와 선택할 때 확인할 조건을 알려줘",
)
pprint(run["response"])

In [ ]:
pprint(run["domain_results"])

## 여러 질문 순차 실행

API 사용량과 rate limit을 고려해 병렬 실행하지 않는다. 필요한 질문만 남긴 뒤 실행한다.

In [ ]:
questions = [
    "IRP 계좌를 다른 금융회사로 이전하려면 어떤 조건을 확인해야 해?",
    "연금 수령 시 세금 판단에 필요한 조건은 무엇이야?",
    "연금 상품의 비용과 위험을 비교할 때 무엇을 확인해야 해?",
]
batch_responses = []
for index, question in enumerate(questions, start=1):
    batch_run = await ask_main_agent(
        question_id=f"NB-BATCH-{index:03d}",
        question=question,
    )
    batch_responses.append(batch_run["response"])

batch_responses

## Runtime 정리

실험을 마치면 HCX, embedding과 Qdrant client를 닫는다. 정리 후 다시 질문하려면 Runtime 조립 셀부터 실행한다.

In [ ]:
await answer_service.aclose()
answer_service = None
"Agent Runtime 종료 완료"